# Gold Headline Sentiment Classifier

### 1.1 Import Libraries and Ignore Warnings

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

### 1.2 Configuration

* **ProsusAI/finbert** is the heavyweight financial expert that gives the highest accuracy because it was trained on market data, but it is slow and heavy for your environment.

* **distilbert-base-uncased** is the balanced "all-rounder" that runs 60% faster than the big models while still being smart enough to handle general text very well.

* **prajjwal1/bert-tiny** is the ultra-lightweight survivor that sacrifices some nuance to be small enough to run on a toaster, making it the perfect choice for limited code space.

In [ ]:
MODEL_NAME = "prajjwal1/bert-tiny" 
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 2e-5
MAX_LEN = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

### 1.3 Load and Fix Data

In [ ]:
def load_data(file_path):
    df = pd.read_csv(file_path)
    df = df[df['Price Sentiment'].isin(['positive', 'negative'])].copy()
    label_map = {'negative': 0, 'positive': 1}
    df['label'] = df['Price Sentiment'].map(label_map)
    return df['News'].values, df['label'].values

texts, labels = load_data('gold-dataset.csv')
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.1, random_state=42)

In [ ]:





# --- 2. Load Data ---


# Load dataset (Ensure file exists)


# --- 3. Dataset Class ---
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

# --- 4. Initialize Components ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = NewsDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = NewsDataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=0, 
    num_training_steps=len(train_loader) * EPOCHS
)

# --- 5. Training Loop ---
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        
    return correct / len(loader.dataset), total_loss / len(loader)

print("Starting training...")
for epoch in range(EPOCHS):
    acc, loss = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Acc: {acc:.4f}")

# --- 6. Prediction with Uncertainty Logic ---
def predict_with_uncertainty(text):
    model.eval()
    encoded = tokenizer.encode_plus(
        text, 
        max_length=MAX_LEN, 
        padding='max_length', 
        truncation=True, 
        return_tensors='pt'
    )
    
    ids = encoded['input_ids'].to(DEVICE)
    mask = encoded['attention_mask'].to(DEVICE)

    with torch.no_grad():
        outputs = model(ids, attention_mask=mask)
        logits = outputs.logits
        # Convert raw scores to probabilities (percentages)
        probs = F.softmax(logits, dim=1)
        
        # Get the highest probability and its index
        max_prob, pred_idx = torch.max(probs, dim=1)
        
        confidence = max_prob.item()
        label = "positive" if pred_idx.item() == 1 else "negative"
        
        if confidence < CONFIDENCE_THRESHOLD:
            return f"The predicted Sentiment for the sentence \"{user_input}\": {label} with {confidence:.0%} confidence. The model is uncertain about this prediction."
        else:
            return f"The predicted Sentiment for the sentence \"{user_input}\": {label} with {confidence:.0%} confidence."

# --- 7. Interactive User Loop ---
print("\n--- Model Ready ---")
print("Enter a news title to check sentiment. Type 'quit' to exit.")

while True:
    user_input = input("\nNews Title: ")
    if user_input.lower() == 'quit':
        print("Exiting...")
        break
    
    result = predict_with_uncertainty(user_input)
    print(result)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Model: prajjwal1/bert-tiny


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training...
Epoch 1/5 | Loss: 0.5972 | Acc: 0.6749
Epoch 2/5 | Loss: 0.2355 | Acc: 0.9202
Epoch 3/5 | Loss: 0.1728 | Acc: 0.9431
Epoch 4/5 | Loss: 0.1548 | Acc: 0.9470
Epoch 5/5 | Loss: 0.1397 | Acc: 0.9556

--- Model Ready ---
Enter a news title to check sentiment. Type 'quit' to exit.
Confidence: 88% -> positive
Confidence: 85% -> negative
Confidence: 92% -> negative
Confidence: 98% -> positive
Confidence: 98% -> positive
Confidence: 88% -> positive
I'm not completely sure (only 70% confident), but I think it is positive.
Confidence: 81% -> positive
Exiting...
